# Dasar-Dasar Pembelajaran Mesin

## Bagian 1, Bagian a: Membaca Data 

### Tujuan Pembelajaran

 - Membuat koneksi basis data SQL ke basis data SQL contoh dan membaca rekaman dari basis data tersebut
 - Mengeksplorasi parameter masukan yang umum

### Paket

 - [Pandas](https://pandas.pydata.org/pandas-docs/stable/)
 - [Pandas.read_sql](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_sql.html)
 - [SQLite3](https://docs.python.org/3.6/library/sqlite3.html)

## Pembacaan data sederhana

Structured Query Language (SQL) adalah [spesifikasi ANSI](https://docs.oracle.com/database/121/SQLRF/ap_standard_sql001.htm#SQLRF55514) yang diimplementasikan oleh berbagai basis data. SQL merupakan format yang andal untuk berinteraksi dengan basis data berukuran besar secara efisien dan memberikan pengalaman yang konsisten di berbagai basis data. Dalam contoh ini kita akan menggunakan sqlite, yaitu versi SQL yang ringan dan agak terbatas. sqlite menggunakan versi SQL yang sedikit dimodifikasi sehingga mungkin berbeda dari yang biasa Anda gunakan. 

In [1]:
# Impor
import sqlite3 as sq3
import pandas.io.sql as pds
import pandas as pd

### Koneksi basis data

Langkah pertama adalah membuat koneksi ke basis data SQL kita. Beberapa basis data SQL yang umum digunakan bersama Python meliputi:

 - Microsoft SQL Server
 - Postgres
 - MySQL
 - AWS Redshift
 - AWS Aurora
 - Oracle DB
 - Terradata
 - Db2 Family
 - Dan masih banyak lainnya
 
Masing-masing basis data ini memerlukan konfigurasi yang sedikit berbeda dan mungkin memerlukan kredensial (nama pengguna dan kata sandi), token, atau persyaratan akses lainnya. Kita akan menggunakan `sqlite3` untuk terhubung ke basis data, tetapi paket koneksi lainnya meliputi:

 - [`SQLAlchemy`](https://www.sqlalchemy.org/) (yang paling umum)
 - [`psycopg2`](http://initd.org/psycopg/)
 - [`MySQLdb`](http://mysql-python.sourceforge.net/MySQLdb.html)

In [2]:
# Inisialisasi jalur ke basis data SQLite
path = 'data/classic_rock.db'
con = sq3.Connection(path)

# Kini kita memiliki koneksi aktif ke basis data SQL

### Membaca data

Setelah terhubung ke basis data, kita dapat menjalankan kueri dan memuat hasilnya sebagai DataFrame Pandas

In [3]:
# Tulis kueri
query = '''
SELECT * 
FROM rock_songs;
'''

# Jalankan kueri
observations = pds.read_sql(query, con)

observations.head()

,Song,Artist,Release_Year,PlayCount
0,Caught Up in You,.38 Special,1982.0,82
1,Hold On Loosely,.38 Special,1981.0,85
2,Rockin' Into the Night,.38 Special,1980.0,18
3,Art For Arts Sake,10cc,1975.0,1
4,Kryptonite,3 Doors Down,2000.0,13


In [4]:
# Kita juga dapat menjalankan kueri SQL apa pun yang didukung
# Tulis kueri
query = '''
SELECT Artist, Release_Year, COUNT(*) AS num_songs, AVG(PlayCount) AS avg_plays  
    FROM rock_songs
    GROUP BY Artist, Release_Year
    ORDER BY num_songs desc;
'''

# Jalankan kueri
observations = pds.read_sql(query, con)

observations.head()

,Artist,Release_Year,num_songs,avg_plays
0,The Beatles,1967.0,23,6.565217
1,Led Zeppelin,1969.0,18,21.000000
2,The Beatles,1965.0,15,3.800000
3,The Beatles,1968.0,13,13.000000
4,The Beatles,1969.0,13,15.000000


## Parameter umum

Ada sejumlah parameter umum yang dapat digunakan untuk membaca data SQL dengan pemformatan tertentu:

 - coerce_float: Mencoba mengubah angka menjadi float
 - parse_dates: Daftar kolom yang akan diurai sebagai tanggal
 - chunksize: Jumlah baris yang disertakan dalam setiap potongan
 
Mari kita lihat penggunaan beberapa parameter tersebut

In [7]:
query='''
SELECT Artist, Release_Year, COUNT(*) AS num_songs, AVG(PlayCount) AS avg_plays  
    FROM rock_songs
    GROUP BY Artist, Release_Year
    ORDER BY num_songs desc;
'''

# Jalankan kueri
observations_generator = pds.read_sql(query,
                            con,
                            coerce_float=True, # Tidak memengaruhi dataset ini karena float telah diurai dengan benar
                            parse_dates=['Release_Year'], # Urai `Release_Year` sebagai tanggal
                            chunksize=5 # Memungkinkan hasil dialirkan sebagai serangkaian tabel yang lebih pendek
                           )

for index, observations in enumerate(observations_generator):
    if index < 5:
        print(f'Observations index: {index}'.format(index))
        display(observations)

Observations index: 0


,Artist,Release_Year,num_songs,avg_plays
0,The Beatles,1970-01-01 00:32:47,23,6.565217
1,Led Zeppelin,1970-01-01 00:32:49,18,21.000000
2,The Beatles,1970-01-01 00:32:45,15,3.800000
3,The Beatles,1970-01-01 00:32:48,13,13.000000
4,The Beatles,1970-01-01 00:32:49,13,15.000000


Observations index: 1


,Artist,Release_Year,num_songs,avg_plays
0,Led Zeppelin,1970-01-01 00:32:50,12,13.166667
1,Led Zeppelin,1970-01-01 00:32:55,12,14.166667
2,Pink Floyd,1970-01-01 00:32:59,11,41.454545
3,Pink Floyd,1970-01-01 00:32:53,10,29.100000
4,The Doors,1970-01-01 00:32:47,10,28.900000


Observations index: 2


,Artist,Release_Year,num_songs,avg_plays
0,Fleetwood Mac,1970-01-01 00:32:57,9,35.666667
1,Jimi Hendrix,1970-01-01 00:32:47,9,24.888889
2,The Beatles,1970-01-01 00:32:43,9,2.444444
3,The Beatles,1970-01-01 00:32:44,9,3.111111
4,Elton John,1970-01-01 00:32:53,8,18.500000


Observations index: 3


,Artist,Release_Year,num_songs,avg_plays
0,Led Zeppelin,1970-01-01 00:32:51,8,47.750000
1,Led Zeppelin,1970-01-01 00:32:53,8,34.125000
2,Boston,1970-01-01 00:32:56,7,69.285714
3,Rolling Stones,1970-01-01 00:32:49,7,36.142857
4,Van Halen,1970-01-01 00:32:58,7,51.142857


Observations index: 4


,Artist,Release_Year,num_songs,avg_plays
0,Bruce Springsteen,1970-01-01 00:32:55,6,7.666667
1,Bruce Springsteen,1970-01-01 00:33:04,6,11.500000
2,Creedence Clearwater Revival,1970-01-01 00:32:49,6,23.833333
3,Creedence Clearwater Revival,1970-01-01 00:32:50,6,18.833333
4,Def Leppard,1970-01-01 00:33:07,6,32.000000
